# Harvest readiness, second test: two white-oyster farms

**What changed since the first notebook.** The first test compared abhi (pink oysters on buckets in a grow tent) with angelo (white oysters on plastic bags). The model failed on the other farm (scores near 0). The two sets differed in species, container and camera distance. This notebook:

1. drops the pink set (the planner is for white oysters, *P. ostreatus*),
2. adds **Duman 2024** (white *P. ostreatus* in black grow bags, greenhouse in Turkey, CC BY 4.0),
3. uses **rectangles** around each cluster instead of traced outlines, because Duman only has rectangles,
4. first shows the dataset's own labels, so we can check them by eye before trusting any score.

| Set | Source | Photos | Their labels | Our label |
|---|---|---|---|---|
| angelo | Roboflow angelo-neyra / mushroom-w7ucu | ~720 | NOT_READY / READY, OVERDUE | young / ready |
| duman | Mendeley hf55tkx489 (Duman et al. 2024) | 555 | Immature / Mature | young / ready |

**The honest test.** Run A trains on Duman and is scored on all angelo photos; run B does the opposite.

**Before you press Run All (once):**
1. Upload Duman as a private Kaggle dataset (see the box below).
2. Right panel, *Session options*: Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
3. *Add-ons, Secrets*: switch `ROBOFLOW_API_KEY` on for this notebook (same secret as last time).
4. *Add Input* (right panel): add your Duman dataset.

Runtime: roughly 30 to 60 minutes. At the end, one small table sums up everything; a screenshot of it is enough.

### Uploading Duman to Kaggle (one time)
On kaggle.com: **Create, New Dataset**, title `duman-oyster-maturity`, keep it **Private**. Drag in these two folders from `mycodemo/data/external/d010_duman2024/`:

* `Raw Images` (555 photos, 270 MB)
* `Maturity Labelled Images and Annotation Files/YOLO Format` (one small .txt label file per photo)

Press **Create**. The notebook finds the two folders by what is inside them (not by their names), so renamed or flattened folders are fine.

In [ ]:
# 1. Install the two libraries we need (Kaggle has the rest)
!pip install -q roboflow ultralytics

In [ ]:
# 2. Settings: change these lines only if you want to
ANGELO = ("angelo-neyra-9rg4y", "mushroom-w7ucu",
          {"not_ready": "young", "ready": "ready", "overdue": "ready"})
DUMAN_LABELS = {"mature": "ready", "immature": "young"}
OUR_LABELS = ["young", "ready"]   # class 0 = young, class 1 = ready
EPOCHS = 50                       # passes over the training photos
IMG_SIZE = 960                    # bigger than last time: Duman clusters are small in a wide photo
MODEL = "yolo11n.pt"              # smallest rectangle model
SEED = 0
WORK = "/kaggle/working"

In [ ]:
# 3. Find the Duman folders you uploaded (by content, not by folder name)
import os, glob, collections
PHOTO_EXT = (".jpg", ".jpeg")

def list_images(d):
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(PHOTO_EXT))

def show_tree():
    for root, dirs, files in sorted(os.walk("/kaggle/input")):
        kinds = collections.Counter(os.path.splitext(f)[1].lower() or "(no extension)" for f in files)
        depth = root.count(os.sep) - 2
        print("  " * depth + os.path.basename(root) + "/", dict(kinds) if kinds else "")

def find_dir():
    best_photos, best_labels = (0, None), (0, None)
    for root, dirs, files in os.walk("/kaggle/input"):
        n_img = sum(f.lower().endswith(PHOTO_EXT) for f in files)
        if n_img > best_photos[0]:
            best_photos = (n_img, root)
        if "classes.txt" in files:
            names = [n.lower() for n in open(os.path.join(root, "classes.txt"), errors="ignore").read().split()]
            n_txt = sum(f.lower().endswith(".txt") for f in files)
            # the Duman maturity labels list Mature and Immature; the other YOLO folder does not
            if "mature" in names and "immature" in names and n_txt > best_labels[0]:
                best_labels = (n_txt, root)
    if best_photos[1] is None or best_labels[0] < 10:
        print("Could not find the Duman photos and labels. What is under /kaggle/input:")
        show_tree()
        raise FileNotFoundError("Duman photos and/or labels not found under /kaggle/input. "
                                "Attach the Duman dataset to this notebook with Add Input (right panel), then run again.")
    return best_photos[1], best_labels[1]

DUMAN_IMG, DUMAN_LAB = find_dir()
print("photos:", DUMAN_IMG, len(list_images(DUMAN_IMG)))
print("labels:", DUMAN_LAB, len(glob.glob(DUMAN_LAB + "/*.txt")))
print("their label names:", open(DUMAN_LAB + "/classes.txt").read().split())

In [ ]:
# 4. Download angelo from Roboflow (version with the fewest photos = no flipped copies)
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
proj = rf.workspace(ANGELO[0]).project(ANGELO[1])
vers = []
for v in proj.versions():
    n = getattr(v, "images", None)
    if not n:
        splits = getattr(v, "splits", None) or {}
        n = sum(int(x) for x in splits.values()) if splits else 10**9
    vers.append((int(n), int(str(v.version).split("/")[-1])))
print("versions (photos, version):", sorted(vers))
ver = sorted(vers)[0][1]
ANGELO_DL = proj.version(ver).download("yolov11", location=f"{WORK}/raw_angelo", overwrite=True).location
print("chosen version", ver, "->", ANGELO_DL)

## Put both sets into the same shape
Every label is rewritten as 0 = young, 1 = ready. angelo's traced outlines are turned into the smallest rectangle around them. Labels that are not in the tables above are dropped and counted, so you can see what was thrown away.

In [ ]:
import shutil, random, yaml, collections

def to_box(parts):
    # 'class x y w h' stays; 'class x1 y1 x2 y2 ...' (outline) becomes its surrounding rectangle
    nums = [float(p) for p in parts[1:]]
    if len(nums) == 4:
        return nums
    xs, ys = nums[0::2], nums[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0]

def pool(short, pairs, their, mapping):
    # pairs = list of (image path, label path); their = list of their label names
    new_id = {i: OUR_LABELS.index(mapping[n.strip().lower()])
              for i, n in enumerate(their) if n.strip().lower() in mapping}
    out = f"{WORK}/pool_{short}"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(f"{out}/images"); os.makedirs(f"{out}/labels")
    counts, dropped = collections.Counter(), collections.Counter()
    for k, (img, lab) in enumerate(pairs):
        lines = []
        if lab and os.path.exists(lab):
            for line in open(lab):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                if c in new_id:
                    lines.append(" ".join([str(new_id[c])] + [f"{v:.6f}" for v in to_box(parts)]))
                    counts[OUR_LABELS[new_id[c]]] += 1
                else:
                    dropped[their[c] if c < len(their) else f"unknown class {c}"] += 1
        name = f"{short}_{k:05d}"
        shutil.copy(img, f"{out}/images/{name}{os.path.splitext(img)[1]}")
        open(f"{out}/labels/{name}.txt", "w").write("\n".join(lines))
    print(f"{short}: {len(pairs)} photos, clusters {dict(counts)}, dropped {dict(dropped)}, "
          f"{sum(counts.values()) / max(len(pairs), 1):.1f} clusters per photo")
    return out

# angelo: pool train/valid/test together (the test photos come from the other farm anyway)
names = yaml.safe_load(open(f"{ANGELO_DL}/data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
ang_pairs = []
for split in ["train", "valid", "test"]:
    for img in sorted(glob.glob(f"{ANGELO_DL}/{split}/images/*")):
        base = os.path.splitext(os.path.basename(img))[0]
        ang_pairs.append((img, f"{ANGELO_DL}/{split}/labels/{base}.txt"))

# duman: one label file per raw photo, same file name
dum_names = open(DUMAN_LAB + "/classes.txt").read().split()
dum_pairs = [(img, f"{DUMAN_LAB}/{os.path.splitext(os.path.basename(img))[0]}.txt")
             for img in list_images(DUMAN_IMG)]

POOL = {"angelo": pool("angelo", ang_pairs, names, ANGELO[2]),
        "duman": pool("duman", dum_pairs, dum_names, DUMAN_LABELS)}

## Data check: what do the photos and labels look like?
Before training we look at the numbers. Very different photo sizes between the two farms, or tiny clusters, can explain a bad score on their own. A strong imbalance between young and ready can bias the model towards the more common label.

In [ ]:
# Data check on the pooled sets (labels are already 0 = young, 1 = ready)
import numpy as np, pandas as pd

def describe(short):
    imgs = sorted(glob.glob(f"{POOL[short]}/images/*"))
    sizes, modes, per_photo, shares = collections.Counter(), collections.Counter(), [], []
    labels = collections.Counter()
    for img in imgs:
        with Image.open(img) as im:
            sizes[f"{im.size[0]}x{im.size[1]}"] += 1
            modes[im.mode] += 1
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        rows = [l.split() for l in open(lab).read().splitlines() if l.strip()]
        per_photo.append(len(rows))
        for c, x, y, w, h in rows:
            labels[OUR_LABELS[int(c)]] += 1
            shares.append(float(w) * float(h) * 100)
    total = max(sum(labels.values()), 1)
    per_photo, shares = np.array(per_photo), np.array(shares)
    fmt = lambda cnt: ", ".join(f"{k}: {v}" for k, v in cnt.most_common())
    rows = [("photos", len(imgs)),
            ("photo sizes (width x height)", fmt(sizes)),
            ("colour channels (PIL mode)", fmt(modes))]
    rows += [(f"clusters, {lab}", f"{labels[lab]} ({100 * labels[lab] / total:.0f}%)") for lab in OUR_LABELS]
    rows += [("photos with no clusters", int((per_photo == 0).sum())),
             ("clusters per photo (median, max)", f"{np.median(per_photo):.0f}, {per_photo.max()}"),
             ("cluster size, % of photo (median, 10th percentile)",
              f"{np.median(shares):.1f}%, {np.percentile(shares, 10):.1f}%" if len(shares) else "n/a")]
    return pd.DataFrame(rows, columns=["check", short]).set_index("check")

for s in POOL:
    print(describe(s).to_string(), "\n")

## Label check: look before you trust
Six random photos from each set with the **dataset's own labels** drawn on (not the model's). Blue = young, orange = ready. Check by eye:
* Does each rectangle hold one whole cluster, or single caps?
* Do the two sets agree on what counts as ready?

If the labels look wrong here, no score below can be trusted.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
COLOURS = {0: "tab:blue", 1: "tab:orange"}

def show_labels(short, n=6, seed=1):
    imgs = sorted(glob.glob(f"{POOL[short]}/images/*"))
    random.seed(seed)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, img in zip(axes.flat, random.sample(imgs, min(n, len(imgs)))):
        im = Image.open(img); W, H = im.size
        ax.imshow(im); ax.axis("off")
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        for line in open(lab).read().splitlines():
            c, x, y, w, h = line.split(); c = int(c)
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False,
                                           edgecolor=COLOURS[c], linewidth=2))
    fig.suptitle(f"{short}: dataset's own labels (blue = young, orange = ready)", fontsize=14)
    plt.tight_layout(); plt.show()

for s in POOL:
    show_labels(s)

In [ ]:
# 5. Settings files for the two runs.
#    Training farm: 85% of its photos to learn from, 15% to check progress.
#    Test farm: ALL of its photos, never seen in training.
def make_run(train_set, test_set):
    random.seed(SEED)
    imgs = sorted(glob.glob(f"{POOL[train_set]}/images/*"))
    random.shuffle(imgs)
    cut = int(0.85 * len(imgs))
    run = f"{WORK}/run_{train_set}_to_{test_set}"
    os.makedirs(run, exist_ok=True)
    open(f"{run}/train.txt", "w").write("\n".join(imgs[:cut]))
    open(f"{run}/val.txt", "w").write("\n".join(imgs[cut:]))
    open(f"{run}/test.txt", "w").write("\n".join(sorted(glob.glob(f"{POOL[test_set]}/images/*"))))
    cfg = {"path": run, "train": f"{run}/train.txt", "val": f"{run}/val.txt",
           "test": f"{run}/test.txt", "names": {0: "young", 1: "ready"}}
    yaml.safe_dump(cfg, open(f"{run}/data.yaml", "w"))
    return run

RUNS = {"A: train duman, test angelo": make_run("duman", "angelo"),
        "B: train angelo, test duman": make_run("angelo", "duman")}

## Train and score
The score is **mAP50 (box)**: roughly, the share of clusters the model puts a rectangle on in the right place *and* with the right label, from 0 (useless) to 1 (perfect). Rough guide: above 0.5 on the unseen farm is a usable first model; below 0.3 means the farms still look too different.

Note: Duman photos come from two fixed cameras, so many photos are near-copies of each other. Its same-farm score will look too good; the other-farm score is the one to trust.

In [ ]:
from ultralytics import YOLO
import pandas as pd
results = []
for name, run in RUNS.items():
    print("\n#####", name)
    model = YOLO(MODEL)
    model.train(data=f"{run}/data.yaml", epochs=EPOCHS, imgsz=IMG_SIZE, seed=SEED,
                project=run, name="train", exist_ok=True, verbose=False, plots=True)
    best = YOLO(f"{run}/train/weights/best.pt")
    own = best.val(data=f"{run}/data.yaml", split="val", imgsz=IMG_SIZE, project=run, name="val_own", exist_ok=True)
    other = best.val(data=f"{run}/data.yaml", split="test", imgsz=IMG_SIZE, project=run, name="test_other", exist_ok=True)
    for where, r in [("same farm (held-back 15%)", own), ("OTHER farm (honest score)", other)]:
        row = {"run": name, "scored on": where, "mAP50 box, all": round(r.box.map50, 3)}
        for i, lab in enumerate(OUR_LABELS):
            try:
                row[f"mAP50 box, {lab}"] = round(r.box.class_result(i)[2], 3)
            except Exception:
                row[f"mAP50 box, {lab}"] = None
        results.append(row)

In [ ]:
# 6. Look at 6 photos from the OTHER farm: dataset's labels (left) vs model's guess (right)
for name, run in RUNS.items():
    best = YOLO(f"{run}/train/weights/best.pt")
    test_imgs = open(f"{run}/test.txt").read().split()
    random.seed(1)
    pick = random.sample(test_imgs, min(3, len(test_imgs)))
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    for (axl, axr), img in zip(axes, pick):
        im = Image.open(img); W, H = im.size
        axl.imshow(im); axl.axis("off"); axl.set_title("dataset's labels")
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        for line in open(lab).read().splitlines():
            c, x, y, w, h = line.split(); c = int(c)
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            axl.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False,
                                            edgecolor=COLOURS[c], linewidth=2))
        axr.imshow(best.predict(img, imgsz=IMG_SIZE, verbose=False)[0].plot()[:, :, ::-1])
        axr.axis("off"); axr.set_title("model's guess")
    fig.suptitle(name, fontsize=14)
    plt.tight_layout(); plt.show()

In [ ]:
# 7. Keep the models and the score table (Output tab), remove the big photo folders
for name, run in RUNS.items():
    shutil.copy(f"{run}/train/weights/best.pt", f"{WORK}/harvest_readiness_{run.split('run_')[-1]}.pt")
for d in glob.glob(f"{WORK}/raw_*") + glob.glob(f"{WORK}/pool_*"):
    shutil.rmtree(d, ignore_errors=True)
summary = pd.DataFrame(results)
summary.to_csv(f"{WORK}/harvest_readiness_scores_2.csv", index=False)
print("\nSCORE TABLE (screenshot this):")
summary

## Read the result
* **Same farm vs OTHER farm.** The OTHER-farm score is the one to trust.
* **young vs ready.** If one label scores much lower, the farms draw the line between young and ready differently.
* **Still near 0?** Then the next step is training on both farms together and testing on a third white-oyster set, or on your own syldd photos once they fruit.